# 07 — Execute and retrieve cloud Jupyter notebooks
Run this local Jupyter notebook after notebook 00 prepares the authorised private cluster. Analysis is executed by a Jupyter kernel on Dataproc; no Spark-submit or local Spark analysis is used. Each cloud notebook checkpoints its outputs after every cell. Inspect returned status/output and Spark UI before advancing to the next notebook.

In [ ]:
from pathlib import Path
import json
import os
import re
import shlex
import subprocess
from IPython.display import display
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
runtime = json.loads((REPO_ROOT / "artifacts" / "execution" / "runtime.json").read_text())
PROJECT_ID, BUCKET, RUN_ID = runtime["project_id"], runtime["bucket"], runtime["run_id"]
REGION, CLUSTER = "asia-southeast1", "dsa5208-rainfall"
if runtime["billing_account_id"] != "01DAD8-AE5F1A-E3910A" or runtime["credit_id"] != "3LU63BPGY77JFDQQ":
    raise RuntimeError("Unexpected billing/credit in runtime manifest.")
def sdk(args, *, as_json=False):
    result = subprocess.run(["gcloud", *args], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+", "[redacted identity]", result.stderr.strip()))
    return json.loads(result.stdout) if as_json else result.stdout
billing = sdk(["billing", "projects", "describe", PROJECT_ID, "--format=json"], as_json=True)
if not billing.get("billingEnabled") or billing.get("billingAccountName") != "billingAccounts/01DAD8-AE5F1A-E3910A":
    raise RuntimeError("Education billing linkage changed; stop execution.")
cluster = sdk(["dataproc", "clusters", "describe", CLUSTER, f"--project={PROJECT_ID}", f"--region={REGION}", "--format=json"], as_json=True)
display({"state": cluster["status"]["state"], "component_urls": cluster["config"].get("endpointConfig", {}).get("httpPorts", {})})
if cluster["status"]["state"] != "RUNNING":
    raise RuntimeError("Cluster is not RUNNING; wait for provisioning and rerun this notebook.")
gce = cluster["config"]["gceClusterConfig"]
if gce.get("serviceAccount") != f"rainfall-vm@{PROJECT_ID}.iam.gserviceaccount.com" or not gce.get("internalIpOnly"):
    raise RuntimeError("Unexpected VM identity or public network configuration.")
ZONE = gce["zoneUri"].rsplit("/", 1)[-1]
MASTER = cluster["config"]["masterConfig"]["instanceNames"][0]
def ssh(command):
    return sdk(["compute", "ssh", MASTER, f"--zone={ZONE}", f"--project={PROJECT_ID}", "--tunnel-through-iap", "--quiet", f"--command={command}"])
print({"run_id": RUN_ID, "zone": ZONE, "master": MASTER})

NOTEBOOK_NAME = os.environ.get("CLOUD_NOTEBOOK", "01_data_quality.ipynb")
START_NOTEBOOK = os.environ.get("START_CLOUD_NOTEBOOK", "false").lower() == "true"


In [ ]:
if START_NOTEBOOK:
    # Bootstrap the verified source bundle on the master.
    extract_code = """from pathlib import Path
    import hashlib, zipfile
    root = Path.home() / 'rainfall-project'
    bundle = root / 'source_bundle.zip'
    assert hashlib.sha256(bundle.read_bytes()).hexdigest() == 'EXPECTED_HASH', 'Bundle hash mismatch'
    with zipfile.ZipFile(bundle) as archive:
        for name in archive.namelist():
            assert (root / name).resolve().is_relative_to(root.resolve()), 'Unsafe archive member'
        archive.extractall(root)
    print('Verified project bundle extracted.')
    """.replace("EXPECTED_HASH", runtime["source_bundle_sha256"])
    command = '\n'.join(["set -eu", 'mkdir -p "$HOME/rainfall-project/executed"', f'gcloud storage cp {shlex.quote(f"gs://{BUCKET}/code/source_bundle.zip")} "$HOME/rainfall-project/source_bundle.zip"', "python3 -c " + shlex.quote(extract_code), '/opt/conda/default/bin/python -c "import nbclient, nbformat, pyspark; print(nbclient.__version__, nbformat.__version__, pyspark.__version__)"'])
    print(ssh(command))
else:
    print("Remote bootstrap skipped during checkpoint retrieval.")


In [ ]:
# Change only after reviewing the preceding executed notebook and Spark jobs.
if NOTEBOOK_NAME not in {"01_data_quality.ipynb", "02_task1_weather_patterns.ipynb", "03_task2_features.ipynb", "04_task2_models.ipynb", "05_final_evaluation.ipynb"}:
    raise ValueError("Unknown analysis notebook.")
cloud_output = f"gs://{BUCKET}/runs/{RUN_ID}/notebooks/{NOTEBOOK_NAME}"
if START_NOTEBOOK:
    # Refuse duplicate launches once a checkpointed cloud output exists.
    exists = subprocess.run(["gcloud", "storage", "objects", "describe", cloud_output, "--format=json"], capture_output=True, text=True)
    if exists.returncode == 0:
        raise RuntimeError("Output already exists; retrieve/review it before any retry.")
    execute_code = f"from scripts.notebook_runner import run_notebook; raise SystemExit(run_notebook('notebooks/{NOTEBOOK_NAME}', 'executed/{NOTEBOOK_NAME}', cloud_output={cloud_output!r}))"
    command = 'cd "$HOME/rainfall-project"; nohup /opt/conda/default/bin/python -c ' + shlex.quote(execute_code) + ' > ' + shlex.quote(f"executed/{NOTEBOOK_NAME}.log") + ' 2>&1 < /dev/null & echo "Cloud Jupyter notebook launched."'
    print(ssh(command))
else:
    print("Launch skipped; use the retrieval cell to inspect the current checkpoint.")


In [ ]:
# Retrieve the latest checkpoint without rerunning any Spark analysis.
import nbformat
local_output = REPO_ROOT / "notebooks" / "executed" / NOTEBOOK_NAME
local_output.parent.mkdir(parents=True, exist_ok=True)
sdk(["storage", "cp", cloud_output, str(local_output)])
executed = nbformat.read(local_output, as_version=4)
display({"notebook": NOTEBOOK_NAME, "status": executed.metadata.get("execution_status"), "started_utc": executed.metadata.get("started_at_utc"), "finished_utc": executed.metadata.get("finished_at_utc"), "executed_cells": [i for i, cell in enumerate(executed.cells) if cell.cell_type == "code" and cell.execution_count is not None]})
for cell in executed.cells:
    for output in cell.get("outputs", []):
        if output.output_type == "error":
            print(output.ename, output.evalue)
print(f"Open the checkpointed notebook to view its full outputs: {local_output}")
